In [1]:
import os

# Grobe Struktur des Analyse-Codes im Repo anschauen
for root, dirs, files in os.walk("src/hot_open"):
    # versteckte Ordner (z.B. __pycache__) überspringen
    dirs[:] = [d for d in dirs if not d.startswith("__")]
    for f in files:
        if f.endswith(".py"):
            print(os.path.join(root, f))

src/hot_open\circular_math.py
src/hot_open\era5_helpers.py
src/hot_open\fastlog_helpers.py
src/hot_open\lidar_helpers.py
src/hot_open\scada_helpers.py
src/hot_open\settings.py
src/hot_open\sourcing_data.py
src/hot_open\unpack.py
src/hot_open\__init__.py


In [2]:
import requests

r = requests.get("https://zenodo.org/records/20204946", timeout=10)
print(r.status_code)

200


In [7]:
import sys; sys.path.insert(0, "C:/Users/ryan.feusi/hill-of-towie-open-source-analysis/src")

In [9]:
import hot_open
print(hot_open.__file__)

C:\Users/ryan.feusi/OneDrive - OST/Dokumente/RES Project/hill-of-towie-open-source-analysis/src\hot_open\__init__.py


In [11]:
from hot_open.sourcing_data import ensure_hot_data_files
from hot_open.settings import get_data_dir
import pandas as pd

# Nur die Turbinen-Metadaten anfordern (kleine Dateien wie Reports werden automatisch mitgeladen)
ensure_hot_data_files(["Hill_of_Towie_turbine_metadata.csv"])

data_dir = get_data_dir()
turbine_metadata = pd.read_csv(data_dir / "Hill_of_Towie_turbine_metadata.csv")
turbine_metadata

,Wind Farm,Turbine Name,Latitude,Longitude,Station ID,Manufacturer,Model,Rated power (kW),Hub Height (m),Rotor Diameter (m),Country,Commercial Operations Date
0,Hill of Towie,T01,57.499214,-3.086743,2304510,Siemens,SWT-2.3-VS-82,2300,59,82,UK,2012-05-12
1,Hill of Towie,T02,57.496266,-3.082818,2304511,Siemens,SWT-2.3-VS-82,2300,59,82,UK,2012-05-12
2,Hill of Towie,T03,57.502070,-3.088980,2304512,Siemens,SWT-2.3-VS-82,2300,59,82,UK,2012-05-12
3,Hill of Towie,T04,57.501967,-3.082085,2304513,Siemens,SWT-2.3-VS-82,2300,59,82,UK,2012-05-12
4,Hill of Towie,T05,57.498911,-3.078124,2304514,Siemens,SWT-2.3-VS-82,2300,59,82,UK,2012-05-12
5,Hill of Towie,T06,57.500245,-3.071322,2304515,Siemens,SWT-2.3-VS-82,2300,59,82,UK,2012-05-12
6,Hill of Towie,T07,57.505133,-3.085851,2304516,Siemens,SWT-2.3-VS-82,2300,59,82,UK,2012-05-12
7,Hill of Towie,T08,57.504656,-3.077542,2304517,Siemens,SWT-2.3-VS-82,2300,59,82,UK,2012-05-12
8,Hill of Towie,T09,57.508278,-3.082573,2304518,Siemens,SWT-2.3-VS-82,2300,59,82,UK,2012-05-12
9,Hill of Towie,T10,57.505425,-3.070656,2304519,Siemens,SWT-2.3-VS-82,2300,59,82,UK,2012-05-12


In [12]:
turbine_metadata.columns.tolist()

['Wind Farm',
 'Turbine Name',
 'Latitude',
 'Longitude',
 'Station ID',
 'Manufacturer',
 'Model',
 'Rated power (kW)',
 'Hub Height (m)',
 'Rotor Diameter (m)',
 'Country',
 'Commercial Operations Date']

In [13]:
turbine_metadata.head()

,Wind Farm,Turbine Name,Latitude,Longitude,Station ID,Manufacturer,Model,Rated power (kW),Hub Height (m),Rotor Diameter (m),Country,Commercial Operations Date
0,Hill of Towie,T01,57.499214,-3.086743,2304510,Siemens,SWT-2.3-VS-82,2300,59,82,UK,2012-05-12
1,Hill of Towie,T02,57.496266,-3.082818,2304511,Siemens,SWT-2.3-VS-82,2300,59,82,UK,2012-05-12
2,Hill of Towie,T03,57.502070,-3.088980,2304512,Siemens,SWT-2.3-VS-82,2300,59,82,UK,2012-05-12
3,Hill of Towie,T04,57.501967,-3.082085,2304513,Siemens,SWT-2.3-VS-82,2300,59,82,UK,2012-05-12
4,Hill of Towie,T05,57.498911,-3.078124,2304514,Siemens,SWT-2.3-VS-82,2300,59,82,UK,2012-05-12


In [14]:
from pyproj import Transformer

# WGS84 (Lat/Lon in Grad) -> UTM Zone 30N (Meter) - passt für Schottland/Hill of Towie
transformer = Transformer.from_crs("EPSG:4326", "EPSG:32630", always_xy=True)

lon = turbine_metadata["Longitude"].to_numpy()
lat = turbine_metadata["Latitude"].to_numpy()
utm_x, utm_y = transformer.transform(lon, lat)

# Ursprung auf die Ecke des Windparks legen -> kleinere, handlichere Zahlen
x = utm_x - utm_x.min()
y = utm_y - utm_y.min()

In [16]:
from py_wake.wind_turbines.generic_wind_turbines import GenericWindTurbine

rated_power_kw = turbine_metadata["Rated power (kW)"].iloc[0]    # 2300 kW
rotor_diameter = turbine_metadata["Rotor Diameter (m)"].iloc[0]  # 82 m
hub_height = turbine_metadata["Hub Height (m)"].iloc[0]          # 59 m

swt23 = GenericWindTurbine(
    name="SWT-2.3-VS-82",
    diameter=rotor_diameter,
    hub_height=hub_height,
    power_norm=rated_power_kw,
    turbulence_intensity=0.1,
)

c:\Users\ryan.feusi\hill-of-towie-open-source-analysis\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [17]:
print(x)
print(y)
print(swt23)
print(swt23.diameter(), swt23.hub_height())

[ 133.68673059  368.53089897    0.          413.18665922  650.20734274
 1058.03935758  187.98687126  685.79090096  384.83949225 1098.54961648
  500.00044178  917.70353501  651.10728806  842.87234405 1566.41628041
 2136.06175886 2468.64348269 2864.47227228 2836.42353397 2909.798893
 3253.49053685]
[ 328.55178717    0.          646.59145257  634.67494141  294.15591122
  442.16612894  987.36860492  933.69822297 1337.27682595 1018.7785494
 1733.98373214 1619.13210564 2270.22472419 1947.03920314  349.64903528
  986.05632221 1150.95849652  891.77879528 1370.3805799  1752.47304198
 1502.52652076]
82.0 59.0


In [18]:
from py_wake.site import UniformWeibullSite
from py_wake.deficit_models.gaussian import BastankhahGaussian

# Platzhalter-Windklima: 12 Sektoren, alle gleich wahrscheinlich, moderate Weibull-Parameter
# -> nur zum Testen der Pipeline, wird gleich durch echte Daten ersetzt
n_sectors = 12
site = UniformWeibullSite(
    p_wd=[1 / n_sectors] * n_sectors,
    a=[9.0] * n_sectors,   # Weibull-Skalierungsparameter (~ mittlere Windgeschwindigkeit)
    k=[2.0] * n_sectors,   # Weibull-Formparameter
    ti=0.1,                # Turbulenzintensität
)

wf_model = BastankhahGaussian(site, swt23)

simulationResult = wf_model(x, y)
aep = simulationResult.aep()
print(f"Gesamt-AEP (Platzhalter-Wind): {aep.sum().values:.1f} GWh")

Gesamt-AEP (Platzhalter-Wind): 154.3 GWh


c:\Users\ryan.feusi\hill-of-towie-open-source-analysis\.venv\Lib\site-packages\py_wake\deficit_models\gaussian.py:124: UserWarning: The BastankhahGaussian model is not representative of the setup used in the literature. For this, use py_wake.literature.gaussian_models.Bastankhah_PorteAgel_2014 instead
  DeprecatedModel.__init__(self, 'py_wake.literature.gaussian_models.Bastankhah_PorteAgel_2014')


In [19]:
from py_wake.literature.gaussian_models import Bastankhah_PorteAgel_2014

wf_model = Bastankhah_PorteAgel_2014(site, swt23, k=0.0324555) # change k later (calculate better k with SCADA-data)

simulationResult = wf_model(x, y)
aep = simulationResult.aep()
print(f"Gesamt-AEP (Platzhalter-Wind, Bastankhah_PorteAgel_2014): {aep.sum().values:.1f} GWh")

Gesamt-AEP (Platzhalter-Wind, Bastankhah_PorteAgel_2014): 153.0 GWh
